In [ ]:
%%shell
# Installs TVM version 0.14.0 from PyPI. If you wish to build
# from source, see https://tvm.apache.org/docs/install/from_source.html
pip install apache-tvm==0.14.0


# Using External Libraries in Relay
**Author**: `Masahiro Masuda <https://github.com/masahi>`_, `Truman Tian <https://github.com/SiNZeRo>`_

This is a short tutorial on how to use external libraries such as cuDNN, or cuBLAS with Relay.

Relay uses TVM internally to generate target specific code. For example, with cuda backend TVM generates cuda kernels for all layers in the user provided network.
But sometimes it is also helpful to incorporate external libraries developed by various vendors into Relay.
Luckily, TVM has a mechanism to transparently call into these libraries.
For Relay users, all we need to do is just to set a target string appropriately.

Before we can use external libraries from Relay, your TVM needs to be built with libraries you want to use.
For example, to use cuDNN, USE_CUDNN option in `cmake/config.cmake` needs to be enabled, and cuDNN include and library directories need to be specified if necessary.

To begin with, we import Relay and TVM.


In [ ]:
import tvm
from tvm import te
import numpy as np
from tvm.contrib import graph_executor as runtime
from tvm import relay
from tvm.relay import testing
import tvm.testing

## Create a simple network
Let's create a very simple network for demonstration.
It consists of convolution, batch normalization, and ReLU activation.



In [ ]:
out_channels = 16
batch_size = 1

data = relay.var("data", relay.TensorType((batch_size, 3, 224, 224), "float32"))
weight = relay.var("weight")
bn_gamma = relay.var("bn_gamma")
bn_beta = relay.var("bn_beta")
bn_mmean = relay.var("bn_mean")
bn_mvar = relay.var("bn_var")

simple_net = relay.nn.conv2d(
    data=data, weight=weight, kernel_size=(3, 3), channels=out_channels, padding=(1, 1)
)
simple_net = relay.nn.batch_norm(simple_net, bn_gamma, bn_beta, bn_mmean, bn_mvar)[0]
simple_net = relay.nn.relu(simple_net)
simple_net = relay.Function(relay.analysis.free_vars(simple_net), simple_net)

data_shape = (batch_size, 3, 224, 224)
net, params = testing.create_workload(simple_net)

## Build and run with cuda backend
We build and run this network with cuda backend, as usual.
By setting the logging level to DEBUG, the result of Relay graph compilation will be dumped as pseudo code.



In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG)  # to dump TVM IR after fusion

target = "cuda"
lib = relay.build_module.build(net, target, params=params)

dev = tvm.device(target, 0)
data = np.random.uniform(-1, 1, size=data_shape).astype("float32")
module = runtime.GraphModule(lib["default"](dev))
module.set_input("data", data)
module.run()
out_shape = (batch_size, out_channels, 224, 224)
out = module.get_output(0, tvm.nd.empty(out_shape))
out_cuda = out.numpy()

The generated pseudo code should look something like below.
Note how bias add, batch normalization, and ReLU activation are fused into the convolution kernel.
TVM generates a single, fused kernel from this representation.



## Use cuDNN for a convolutional layer
We can use cuDNN to replace convolution kernels with cuDNN ones.
To do that, all we need to do is to append the option " -libs=cudnn" to the target string.



In [ ]:
net, params = testing.create_workload(simple_net)
target = "cuda -libs=cudnn"  # use cudnn for convolution
lib = relay.build_module.build(net, target, params=params)

dev = tvm.device(target, 0)
data = np.random.uniform(-1, 1, size=data_shape).astype("float32")
module = runtime.GraphModule(lib["default"](dev))
module.set_input("data", data)
module.run()
out_shape = (batch_size, out_channels, 224, 224)
out = module.get_output(0, tvm.nd.empty(out_shape))
out_cudnn = out.numpy()

Note that if you use cuDNN, Relay cannot fuse convolution with layers following it.
This is because layer fusion happens at the level of TVM internal representation(IR).
Relay treats external libraries as black box, so there is no way to fuse them with TVM IR.

The pseudo code below shows that cuDNN convolution + bias add + batch norm + ReLU turned into two stages of computation, one for cuDNN call and the other for the rest of operations.



## Verify the result
We can check that the results of two runs match.



In [ ]:
tvm.testing.assert_allclose(out_cuda, out_cudnn, rtol=1e-5)

## Conclusion
This tutorial covered the usage of cuDNN with Relay.
We also have support for cuBLAS. If cuBLAS is enabled, it will be used inside a fully connected layer (relay.dense).
To use cuBLAS, set a target string as "cuda -libs=cublas".
You can use both cuDNN and cuBLAS with "cuda -libs=cudnn,cublas".

For ROCm backend, we have support for MIOpen and rocBLAS.
They can be enabled with target "rocm -libs=miopen,rocblas".

Being able to use external libraries is great, but we need to keep in mind some cautions.

First, the use of external libraries may restrict your usage of TVM and Relay.
For example, MIOpen only supports NCHW layout and fp32 data type at the moment, so you cannot use other layouts or data type in TVM.

Second, and more importantly, external libraries restrict the possibility of operator fusion during graph compilation, as shown above.
TVM and Relay aim to achieve the best performance on a variety of hardwares, with joint operator level and graph level optimization.
To achieve this goal, we should continue developing better optimizations for TVM and Relay, while using external libraries as a nice way to fall back to existing implementation when necessary.

